# Efficient TTT: Adaptive Test-Time Training for VQA

**Authors:** Yugesh Sappidi & Aishwarya Reddy Chinthalapudi  
**Course:** CS 518 — Deep Learning for Computer Vision (Prof. Sathya N. Ravi, UIC)

This notebook runs GPU tasks on Google Colab. All scripts are stored on Google Drive.

## Setup (run once per session)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add project to Python path
import sys
PROJECT_DIR = '/content/drive/MyDrive/AdaTTT'
sys.path.insert(0, PROJECT_DIR)

# Install dependencies
!pip install -q torch torchvision transformers datasets pillow pyyaml scipy tqdm

# Check GPU
!nvidia-smi

In [ ]:
# Verify imports
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

from ttt import FullVQAModel, TTTAdapter, AdaptiveRouter
print('\n✅ All imports successful!')

## Step 1: Train Base VQA Model

Trains fusion + gate + prediction head with cross-entropy + auxiliary gate loss.  
**Estimated time:** ~4-6 hours on A100, ~8-10 hours on T4

In [ ]:
!python {PROJECT_DIR}/gpu/train_base.py --config {PROJECT_DIR}/config/config.yaml --epochs 15

## Step 2: Run TTT Sweep

Evaluate with different K values and TTT objectives.  
Run each configuration separately.

In [ ]:
# Baseline (no TTT)
!python {PROJECT_DIR}/gpu/run_ttt_sweep.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
    --k 0

In [ ]:
# K=1, masked_patch
!python {PROJECT_DIR}/gpu/run_ttt_sweep.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
    --k 1 --objective masked_patch

In [ ]:
# K=1, rotation
!python {PROJECT_DIR}/gpu/run_ttt_sweep.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
    --k 1 --objective rotation

In [ ]:
# K=3 and K=5 (masked_patch)
!python {PROJECT_DIR}/gpu/run_ttt_sweep.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
    --k 3 --objective masked_patch

!python {PROJECT_DIR}/gpu/run_ttt_sweep.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
    --k 5 --objective masked_patch

## Step 3: Train Confidence Gate

**Prerequisites:** Run `scripts/04_generate_gate_labels.py` locally first!  
**Estimated time:** ~30 minutes

In [ ]:
!python {PROJECT_DIR}/gpu/train_gate.py \
    --config {PROJECT_DIR}/config/config.yaml \
    --base-checkpoint {PROJECT_DIR}/checkpoints/base/best.pt

## Step 4: Adaptive Inference (Pareto Curve)

Run at each threshold τ to build the Pareto frontier.

In [ ]:
for tau in [0.5, 0.7, 0.8, 0.9, 0.95]:
    !python {PROJECT_DIR}/gpu/run_inference.py \
        --config {PROJECT_DIR}/config/config.yaml \
        --base-checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
        --gate-checkpoint {PROJECT_DIR}/checkpoints/gate/best.pt \
        --threshold {tau} --k 1 --objective masked_patch

## Step 5: Ablation Study

Compare TTT stabilization techniques.

In [ ]:
for mode in ['ttt_only', 'ttt_consistency', 'ttt_mixup', 'ttt_both']:
    !python {PROJECT_DIR}/gpu/run_ablation.py \
        --config {PROJECT_DIR}/config/config.yaml \
        --checkpoint {PROJECT_DIR}/checkpoints/base/best.pt \
        --k 1 --mode {mode}

## Post-Processing (run locally)

After all GPU tasks are complete, run these locally:
```bash
python scripts/02_analyze_results.py --config config/config.yaml
python scripts/03_generate_figures.py --config config/config.yaml
```